[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/soccer-vision/blob/master/examples/eval_pitch.ipynb)

# Pitch-model evaluation (keypoint accuracy vs labeler ground truth)
Scores a trained pitch model on the FROZEN held-out benchmark. Headline:
accurate-coverage = % of frames matching the labeler within its noise floor.
Replaces the old anchor_cov coverage gate (which gave a false pass).

In [ ]:
!pip install -q "soccer-vision @ git+https://github.com/PatrickJReed/soccer-vision.git#subdirectory=packages/soccer-vision"
!pip install -q "ultralytics>=8.2"
from pathlib import Path
WEIGHTS = Path("/content/pitch_v1.pt")
BENCH = Path("/content/benchmark")        # dir with benchmark.json + per-field homographies + clips
if not WEIGHTS.exists() or not BENCH.exists():
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/soccer-vision")
    if not WEIGHTS.exists():
        for cand in ("last.pt", "best.pt", "pitch_v1.pt"):
            if (DRIVE / cand).exists():
                !cp "{DRIVE/cand}" /content/pitch_v1.pt
                print("weights:", cand); break
    if not BENCH.exists() and (DRIVE / "benchmark").exists():
        !cp -r "{DRIVE/'benchmark'}" /content/benchmark
assert WEIGHTS.exists() and (BENCH / "benchmark.json").exists()


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from soccer_vision.eval.benchmark import load_manifest
from soccer_vision.pipeline import homographies_from_parquet

manifest = load_manifest(BENCH / "benchmark.json")
model = YOLO(str(WEIGHTS))
FRAME_SIZE = (1920, 1080)

# gt_homographies/model_predictions keyed by a global (field, frame) id flattened to int
gt_homs, preds, key = {}, {}, 0
keymap = {}
for fld in manifest.fields:
    homs = homographies_from_parquet(BENCH / fld.homographies)
    # the per-field clip is expected at BENCH/<field>.mp4
    cap = cv2.VideoCapture(str(BENCH / f"{fld.field}.mp4"))
    for fi in fld.frame_indices:
        if fi not in homs:
            continue
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi); ok, frame = cap.read()
        if not ok:
            continue
        gt_homs[key] = np.asarray(homs[fi].H, float)
        res = model.predict(frame, imgsz=1280, verbose=False)[0]
        kp = res.keypoints
        if kp is not None and kp.xy is not None and len(kp.xy):
            xy = kp.xy[0].cpu().numpy(); cf = kp.conf[0].cpu().numpy()
            preds[key] = np.column_stack([xy, cf])
        keymap[key] = (fld.field, fi); key += 1
    cap.release()
print(f"benchmark frames with GT: {len(gt_homs)}; with model prediction: {len(preds)}")


In [ ]:
from soccer_vision.eval.pitch_metrics import labeler_fit_residual_feet

# labeler noise floor: median per-frame fit residual (feet), using the keypoints
# parquet clicks if available; else fall back to projecting visible landmarks.
# Patrick: set MARGIN from a test-retest if you ran one, else keep the default.
residuals = []
for fld in manifest.fields:
    homs = homographies_from_parquet(BENCH / fld.homographies)
    kpt_path = BENCH / fld.homographies.replace("homographies", "keypoints")
    if kpt_path.exists():
        import pandas as pd
        kdf = pd.read_parquet(kpt_path)
        for fi in fld.frame_indices:
            sub = kdf[kdf["frame"] == fi]
            if fi in homs and len(sub) >= 1:
                residuals.append(labeler_fit_residual_feet(
                    np.asarray(homs[fi].H, float),
                    sub[["x_px", "y_px"]].to_numpy(float),
                    sub["kp_idx"].to_numpy(int)))
NOISE_FLOOR_FT = float(np.percentile(residuals, 90)) if residuals else 5.0
MARGIN_FT = 1.0
MATCH_THRESHOLD_FT = NOISE_FLOOR_FT + MARGIN_FT
print(f"labeler noise floor (p90): {NOISE_FLOOR_FT:.2f} ft  -> match threshold {MATCH_THRESHOLD_FT:.2f} ft")


In [ ]:
from soccer_vision.eval.pitch_metrics import score_benchmark
from soccer_vision.pitch.landmarks import LANDMARK_NAMES

rep = score_benchmark(gt_homs, preds, frame_size=FRAME_SIZE,
                      match_threshold_feet=MATCH_THRESHOLD_FT)
print(f"=== HEADLINE: accurate-coverage {rep.accurate_coverage:.1%} "
      f"({rep.n_matched}/{rep.n_frames} frames match the labeler) ===")
print(f"keypoint feet error  median {rep.keypoint_feet_median:.2f}  p90 {rep.keypoint_feet_p90:.2f}")
print(f"homography reproj ft median {rep.reproj_feet_median:.2f}  p90 {rep.reproj_feet_p90:.2f}")
print(f"excluded degenerate-GT frames: {rep.n_excluded_degenerate}\n")
print(f"{'idx':>3} {'landmark':22} {'median_ft':>9} {'p90_ft':>8} {'detect':>7}")
for i, s in rep.per_landmark.items():
    print(f"{i:>3} {LANDMARK_NAMES[i]:22} {s['median']:>9.2f} {s['p90']:>8.2f} {s['detect_rate']:>7.0%}")


## Qualitative overlays (Patrick assesses)
The next cell renders model keypoints (green) vs labeler GT keypoints (red) on
sampled benchmark frames and saves a contact sheet. Claude does not interpret it
— review it yourself.

In [ ]:
import cv2, numpy as np
from soccer_vision.pitch.autolabel import project_landmarks
from soccer_vision.pitch.landmarks import PITCH_LANDMARKS, NEAR_HALFWAY_IDX

cells = []
for k in list(gt_homs)[:8]:
    fld, fi = keymap[k]
    cap = cv2.VideoCapture(str(BENCH / f"{fld}.mp4")); cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
    ok, img = cap.read(); cap.release()
    if not ok: continue
    gt = project_landmarks(gt_homs[k], PITCH_LANDMARKS, FRAME_SIZE)
    for i in range(len(PITCH_LANDMARKS)):
        if i == NEAR_HALFWAY_IDX: continue
        if gt[i,2] > 0:
            cv2.circle(img,(int(gt[i,0]),int(gt[i,1])),12,(0,0,255),2)  # GT red ring
    if k in preds:
        m = preds[k]
        for i in range(len(PITCH_LANDMARKS)):
            if i != NEAR_HALFWAY_IDX and m[i,2] >= 0.5:
                cv2.circle(img,(int(m[i,0]),int(m[i,1])),8,(60,220,120),-1)  # model green dot
    cells.append(img)
cw=640
tiles=[cv2.resize(c,(cw,int(c.shape[0]*cw/c.shape[1]))) for c in cells]
rows=[np.hstack(tiles[i:i+4]+[np.zeros_like(tiles[0])]*(4-len(tiles[i:i+4]))) for i in range(0,len(tiles),4)]
cv2.imwrite("/content/eval_overlay.jpg", np.vstack(rows))
from IPython.display import Image, display
display(Image("/content/eval_overlay.jpg"))
